In [74]:
import sympy as sym
from sympy import *
import numpy as np
from tabulate import tabulate

#constants for 11B nuclei
Ispin = 3/2
w0 = 192.55 #Larmor Frequency for 11B (MHz)
whz = w0*10**6

wkhz = w0*10**3 #Larmor freq in kHz

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*(whz))

# Coefficient for LHQ (cluster 1) from ASICS (in Hz)
A_coeff = [-1.730215*10**3,-2.744504*10**3,-3.396561*10**3]
B_coeff = [1.251296*10**3,2.561384*10**3,-1.133846*10**3]
C_coeff = [3.573296*10**3,0.986618*10**3,-0.473004*10**3]
D_coeff = [-0.060956*10**3,-0.376258*10**3,-0.980912*10**3]
E_coeff = [-0.037878*10**3,-0.579924*10**3,-1.166911*10**3]

#Coefficient for LHQ (cluster 2) from ASICS (in Hz)
# A_coeff = [-2.237*10**3,-2.631*10**3,-3.326*10**3]
# B_coeff = [1.436*10**3,2.462*10**3,-1.178*10**3]
# C_coeff = [-3.527*10**3,0.272*10**3,-0.705*10**3]
# D_coeff = [0.223*10**3,-0.402*10**3,-0.926*10**3]
# E_coeff = [0.005*10**3, -0.526*10**3,-1.129*10**3]

print(q, A_coeff, B_coeff, C_coeff)

-3.895092183848351e-09 [-1730.2150000000001, -2744.504, -3396.561] [1251.296, 2561.384, -1133.846] [3573.296, 986.6179999999999, -473.00399999999996]


In [75]:
#Define symbol for quadrupolar tensor and force them to be real
AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q = sym.symbols('AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q', real=True)

#Variable for each equation set
quad_tensor = [(AzzmAyy_Q, Ayz_Q), (AzzmAxx_Q, Axz_Q), (AyymAxx_Q, Axy_Q)]

# List to hold solutions for quadrupolar tensor terms
solutions_Q = []

for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
    
    eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q/8)), D_coeff[i])
    eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q/2), E_coeff[i])

    # Solve the system
    solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
    solutions_Q.append(solution)

# Assign the solutions to the respective variables
AzzmAyy_Q = [solutions_Q[0][0][0], solutions_Q[0][1][0]]
Ayz_Q = [solutions_Q[0][0][1], solutions_Q[0][1][1]]
AzzmAxx_Q = [solutions_Q[1][0][0], solutions_Q[1][1][0]]
Axz_Q = [solutions_Q[1][0][1], solutions_Q[1][1][1]]
AyymAxx_Q = [solutions_Q[2][0][0], solutions_Q[2][1][0]]
Axy_Q = [solutions_Q[2][0][1], solutions_Q[2][1][1]]

# print(solutions_Q)
# Print the final results for the variables
print(f"Azz - Ayy: {AzzmAyy_Q}, Ayz: {Ayz_Q}")
print(f"Azz - Axx: {AzzmAxx_Q}, Axz: {Axz_Q}")
print(f"Ayy - Axx: {AyymAxx_Q}, Axy: {Axy_Q}")  



Azz - Ayy: [-35120.8684028640, 35120.8684028640], Ayz: [-61530.6552120510, 61530.6552120510]
Azz - Axx: [-189595.156285395, 189595.156285395], Axz: [-174507.296397013, 174507.296397013]
Ayy - Axx: [-249031.671571717, 249031.671571717], Axy: [-267333.199332134, 267333.199332134]


In [76]:
import itertools
# Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
combinations = list(itertools.product(AzzmAxx_Q, AyymAxx_Q, AzzmAyy_Q))
print(combinations)

[(-189595.156285395, -249031.671571717, -35120.8684028640), (-189595.156285395, -249031.671571717, 35120.8684028640), (-189595.156285395, 249031.671571717, -35120.8684028640), (-189595.156285395, 249031.671571717, 35120.8684028640), (189595.156285395, -249031.671571717, -35120.8684028640), (189595.156285395, -249031.671571717, 35120.8684028640), (189595.156285395, 249031.671571717, -35120.8684028640), (189595.156285395, 249031.671571717, 35120.8684028640)]


In [77]:

#Find Quadrupolar tensor diagonal elements

Axx1 = []; Axx2 = []; Axx3 = []
Ayy1 = []; Ayy2 = []; Ayy3 = []
Azz1 = []; Azz2 = []; Azz3 = []

# Initialize variables to track the best combination and minimum variation
best_combination = None
min_variation = float('inf')
for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
    # Solution 1
    Axx1_val = (-(AzzmAxx_val + AyymAxx_val)/3)
    Ayy1_val = Axx1_val + AyymAxx_val
    Azz1_val = Axx1_val + AzzmAxx_val

    #Save values
    Axx1.append(Axx1_val)
    Ayy1.append(Ayy1_val)
    Azz1.append(Azz1_val)

     # Solution 2
    Ayy2_val = -(AzzmAyy_val - AyymAxx_val) / 3
    Axx2_val = Ayy2_val - AyymAxx_val
    Azz2_val = Ayy2_val + AzzmAyy_val

     #Save values
    Axx2.append(Axx2_val)
    Ayy2.append(Ayy2_val)
    Azz2.append(Azz2_val)

    # Solution 3
    Azz3_val = (AzzmAxx_val + AzzmAyy_val) / 3
    Axx3_val = Azz3_val - AzzmAxx_val
    Ayy3_val = Azz3_val - AzzmAyy_val
    
    #Save values
    Axx3.append(Axx3_val)
    Ayy3.append(Ayy3_val)
    Azz3.append(Azz3_val)

    # Convert sympy Float to regular Python float for NumPy functions
    Axx1_val = float(Axx1_val)
    Axx2_val = float(Axx2_val)
    Axx3_val = float(Axx3_val)
    
    Ayy1_val = float(Ayy1_val)
    Ayy2_val = float(Ayy2_val)
    Ayy3_val = float(Ayy3_val)
    
    Azz1_val = float(Azz1_val)
    Azz2_val = float(Azz2_val)
    Azz3_val = float(Azz3_val)

    # Calculate variation (standard deviation) for Axx, Ayy, Azz
    variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
    variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
    variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

    total_variation = variation_Axx + variation_Ayy + variation_Azz

    # Update the best combination if the current one has less variation
    if total_variation < min_variation:
        min_variation = total_variation
        best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
        best_Axx_Q = np.mean([Axx1_val, Axx2_val, Axx3_val])
        best_Ayy_Q = np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
        best_Azz_Q = np.mean([Azz1_val, Azz2_val, Azz3_val])

#Get index for off-diagonal elements        
index_AzzmAxx = AzzmAxx_Q.index(best_combination[0])
best_Axz_Q = Axz_Q[index_AzzmAxx]

index_AyymAxx = AyymAxx_Q.index(best_combination[1])
best_Axy_Q = Axy_Q[index_AyymAxx]

index_AzzmAyy = AzzmAyy_Q.index(best_combination[2])
best_Ayz_Q = Ayz_Q[index_AzzmAyy]

# Print results
print("Axx1:", Axx1)
print("Axx2:", Axx2)
print("Axx3:", Axx3)

print("Ayy1:", Ayy1)
print("Ayy2:", Ayy2)
print("Ayy3:", Ayy3)

print("Azz1:", Azz1)
print("Azz2:", Azz2)
print("Azz3:", Azz3)

print("Best combination with minimum standard deviation:")
print("AzzmAxx:", best_combination[0])
print("AyymAxx:", best_combination[1])
print("AzzmAyy:", best_combination[2])
print("Axz:", best_Axz_Q)
print("Axy:", best_Axy_Q)
print("Ayz:", best_Ayz_Q)

print("Average of Axx1, Axx2, Axx3 with minimum standard deviation:", best_Axx_Q)
print("Average of Ayy1, Ayy2, Ayy3 with minimum standard deviation:", best_Ayy_Q)
print("Average of Azz1, Azz2, Azz3 with minimum standard deviation:", best_Azz_Q)


Axx1: [146208.942619037, 146208.942619037, -19812.1717621073, -19812.1717621073, 19812.1717621073, 19812.1717621073, -146208.942619037, -146208.942619037]
Axx2: [177728.070515432, 154314.158246856, -154314.158246856, -177728.070515432, 177728.070515432, 154314.158246856, -154314.158246856, -177728.070515432]
Axx3: [114689.814722642, 138103.726991218, 114689.814722642, 138103.726991218, -138103.726991218, -114689.814722642, -138103.726991218, -114689.814722642]
Ayy1: [-102822.728952680, -102822.728952680, 229219.499809609, 229219.499809609, -229219.499809609, -229219.499809609, 102822.728952680, 102822.728952680]
Ayy2: [-71303.6010562842, -94717.5133248602, 94717.5133248602, 71303.6010562842, -71303.6010562842, -94717.5133248602, 94717.5133248602, 71303.6010562842]
Ayy3: [-39784.4731598889, -86612.2976970409, -39784.4731598889, -86612.2976970409, 86612.2976970409, 39784.4731598889, 86612.2976970409, 39784.4731598889]
Azz1: [-43386.2136663575, -43386.2136663575, -209407.328047502, -20940

In [78]:
#Define symbol for CSA tensor and force them to be real
Azz_s, Axx_s, Ayy_s, Ayz_s, Axz_s, Axy_s = sym.symbols('Azz_s,Axx_s,Ayy_s,Ayz_s,Axz_s,Axy_s', real=True)

#Variables for each equation
cs_tensor = [(Ayy_s, Azz_s, Ayz_s), # for x -> Abb = Ayy; Agg = Azz; Abg = Ayz
             (Axx_s, Azz_s, Axz_s), # for y -> Abb = Axx; Agg = Azz; Abg = Axz
             (Axx_s, Ayy_s, Axy_s)] # for z -> Abb = Axx; Agg = Ayy; Abg = Axy

#Store variables in dictionary for access
A = {
    'xx': best_Axx_Q, 'yy': best_Ayy_Q, 'zz': best_Azz_Q,
    'yz': best_Ayz_Q, 'zy': best_Ayz_Q,
    'xz': best_Axz_Q, 'zx': best_Axz_Q,
    'xy': best_Axy_Q, 'yx': best_Axy_Q,
}

#Define rotation tuple (a, b, g, bg, m)
rotations = [
    ('x', 'y', 'z', 'yz', 1),   # a = x, b = y, g = z, m = 1
    ('y', 'x', 'z', 'xz', 1),  # a = y, b = x, g = z, m = 1
    ('z', 'x', 'y', 'xy', -1)  # a = z, b = x, g = y, m = -1
]

# List to hold solutions
solutions_cs = []

for i, (Abb_s, Agg_s, Abg_s) in enumerate(cs_tensor):
    a, b, g, bg, m = rotations[i]
    eq1 = sym.Eq(
        (8*A[a+a]*(A[b+b] + A[g+g] - A[a+a]) + 16*(A[a+b]**2 + A[a+g]**2) + 5*(A[b+b]**2 + A[g+g]**2) + 28*A[b+g]**2 - 18*A[b+b]*A[g+g])*(q/8) - 0.5*(Abb_s + Agg_s)*whz, A_coeff[i]
        )
    
    eq2 = sym.Eq(
        m*(2*A[a+a]*(A[b+b] - A[g+g]) - 12*(A[a+b]**2 - A[a+g]**2) - A[b+b]**2 + A[g+g]**2)*(q/2) - 0.5*m*(Agg_s - Abb_s)*whz, B_coeff[i]
        )
    
    eq3 = sym.Eq(
        -m*(-2*A[a+a]*A[b+g] + 12*A[a+b]*A[a+g] + A[b+g]*(A[b+b] + A[g+g]))*q - m*Abg_s*whz, C_coeff[i]
    )
    # Solve the system
    solution = sym.solve([eq1, eq2, eq3], (Abb_s, Agg_s, Abg_s))
    solutions_cs.append(solution)

print(solutions_cs)

#saving solutions
Axx_s = np.mean([solutions_cs[1][Axx_s], solutions_cs[2][Axx_s]])
Ayy_s = np.mean([solutions_cs[0][Ayy_s], solutions_cs[2][Ayy_s]])
Azz_s = np.mean([solutions_cs[0][Azz_s], solutions_cs[1][Azz_s]])

Ayz_s = solutions_cs[0][Ayz_s]
Axz_s = solutions_cs[1][Axz_s]
Axy_s = solutions_cs[2][Axy_s]

print('Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: \n', Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s)


[{Ayy_s: 6.86289619187512e-6, Azz_s: 4.20545917436024e-6, Ayz_s: -7.77914061205763e-6}, {Axx_s: 1.32888018076280e-5, Azz_s: 4.24975992636871e-6, Axz_s: -1.01200515561466e-5}, {Axx_s: 1.25634334089429e-5, Ayy_s: 8.01243877630728e-6, Axy_s: -5.89842130865167e-6}]
Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: 
 1.29261176082855e-5 7.43766748409120e-6 4.22760955036447e-6 -7.77914061205763e-6 -1.01200515561466e-5 -5.89842130865167e-6


In [79]:
#Quadrupolar Tensor in Tenon Frame
Q_T = np.zeros((3,3))
Q_T[0,0] = best_Axx_Q; Q_T[0,1] = best_Axy_Q; Q_T[0,2] = best_Axz_Q;
Q_T[1,0] = best_Axy_Q; Q_T[1,1] = best_Ayy_Q; Q_T[1,2] = best_Ayz_Q;
Q_T[2,0] = best_Axz_Q; Q_T[2,1] = best_Ayz_Q; Q_T[2,2] = best_Azz_Q;

print('Quadrupolar tensor (tenon frame): \n', Q_T)

#CSA tensor in tenon frame
CS_T = np.zeros((3,3))
CS_T[0,0] = Axx_s; CS_T[0,1] = Axy_s; CS_T[0,2] = Axz_s;
CS_T[1,0] = Axy_s; CS_T[1,1] = Ayy_s; CS_T[1,2] = Ayz_s;
CS_T[2,0] = Axz_s; CS_T[2,1] = Ayz_s; CS_T[2,2] = Azz_s;

print('Chemical Shift tensor (tenon frame): \n', CS_T)



Quadrupolar tensor (tenon frame): 
 [[ 146208.94261904 -267333.19933213 -174507.29639701]
 [-267333.19933213  -94717.51332486   61530.65521205]
 [-174507.29639701   61530.65521205  -51491.42929418]]
Chemical Shift tensor (tenon frame): 
 [[ 1.29261176e-05 -5.89842131e-06 -1.01200516e-05]
 [-5.89842131e-06  7.43766748e-06 -7.77914061e-06]
 [-1.01200516e-05 -7.77914061e-06  4.22760955e-06]]


In [80]:
#following the Voseggard et al. paper for principal frame parameters JOURNAL OF MAGNETIC RESONANCE, Series A 122, 111 – 119 ( 1996 ) ARTICLE NO. 0186

#Calculate Quadrupolar Tensor in PAS
eigenvalues, eigenvectors = np.linalg.eig(Q_T)
D_quad = np.diag(eigenvalues)
print('Diagonalized Quadrupolar Tensor:\n', D_quad, '\n')

quad_avg = np.mean(eigenvalues) # Tr(A)Quad/3
sorted_eigenvalues = sorted((eigenvalues - quad_avg), key=abs)

print('Sorted Eigenvalues of Quadrupolar diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues, '\n') 

Vyy = (sorted_eigenvalues[0]+ quad_avg)*(2*Ispin*(2*Ispin - 1)) 
Vxx = (sorted_eigenvalues[1] + quad_avg)*(2*Ispin*(2*Ispin - 1))
Vzz = (sorted_eigenvalues[2] + quad_avg)*(2*Ispin*(2*Ispin - 1)) 

print('Quad Tensors Vzz, Vyy, Vxx: \n', Vzz, Vyy, Vxx)

print('================================================================================================')
#Calculate Quadrupolar Tensor in PAS
eigenvalues, eigenvectors = np.linalg.eig(CS_T)
D_cs = np.diag(eigenvalues)
print('Diagonalized CSA Tensor:\n', D_cs, '\n')

cs_avg = np.mean(eigenvalues) # Tr(A)CSA/3
sorted_eigenvalues = sorted((eigenvalues - cs_avg), key=abs)

print('Sorted Eigenvalues of CSA diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues, '\n') 

csyy = -(sorted_eigenvalues[0] + cs_avg) 
csxx = -(sorted_eigenvalues[1] + cs_avg)
cszz = -(sorted_eigenvalues[2] + cs_avg) 

print('Quad Tensors δzz, δyy, δxx: \n', cszz, csyy, csxx)

Diagonalized Quadrupolar Tensor:
 [[ 392392.82567951       0.               0.        ]
 [      0.         -278212.72231354       0.        ]
 [      0.               0.         -114180.10336597]] 

Sorted Eigenvalues of Quadrupolar diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):
 [-114180.10336596849, -278212.7223135423, 392392.82567951083] 

Quad Tensors Vzz, Vyy, Vxx: 
 2354356.954077064 -685080.6201958116 -1669276.3338812548
Diagonalized CSA Tensor:
 [[-8.42548976e-06  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  1.96448082e-05  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.33720762e-05]] 

Sorted Eigenvalues of CSA diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):
 [5.174944666045167e-06, 1.1447676641225652e-05, -1.6622621307270817e-05] 

Quad Tensors δzz, δyy, δxx: 
 8.425489759690436e-06 -1.3372076213625548e-05 -1.9644808188806033e-05


In [81]:
#Quadrupolar tensor parameters
cq = Vzz/10**6
etaq = (Vyy - Vxx)/Vzz

#CSA tensor parameters
iso_cs = np.mean([cszz, csyy, csxx]) #converting Hz to ppm (Should be multiplied by 10**6?)
csa = cszz - iso_cs
etas = (csyy - csxx)/csa


table = [['cq (MHz)', cq], ['etaq', etaq ], ['iso_cs (ppm)', iso_cs], ['csa (ppm)', csa], ['etas', etas] ]
print(tabulate(table, headers=['Quantity', 'Fit Value']))

Quantity         Fit Value
------------  ------------
cq (MHz)       2.35436
etaq           0.418032
iso_cs (ppm)  -8.19713e-06
csa (ppm)      1.66226e-05
etas           0.377361
